In [1]:
"""
Stage 2 (ES vs LS) — Stratified 70/30 train/test split for
vst_stratum_aware_ESLS.csv
=========================================================================

Composite stratification on (condition x cell_type x sex), with
*cell_type* treated as the priority confounder. Sex is used in the
composite when it doesn't create singleton strata (the ES vs LS cohort
is heavily male-biased: M=82, F=20, so the (cond x cell x sex) key
will be checked and gracefully fall back to (cond x cell_type) when a
stratum has n=1).

Three post-split diagnostics:

  (a) Chi-squared train-vs-test contingency tests on each stratification
      variable + a multi-panel visualisation (marginal proportions,
      composite stacked bars, summary table, cross-tab heatmaps).

  (b) Kolmogorov-Smirnov tests on the leading principal components of
      train vs test, where the PCA is fitted on TRAIN ONLY and applied
      to test.

  (c) Variance partitioning per gene (partial R^2 for condition,
      cell_type, sex, and unexplained residual) BEFORE vs AFTER mirror
      OLS residualization. "Mirror" = OLS coefficients are estimated on
      TRAIN ONLY and applied verbatim to TEST.

Strict no-leakage rules applied throughout
------------------------------------------
* PCA is fitted on X_train only; test scores come from .transform().
* OLS confounder regression is fitted on X_train only; test residuals
  come from applying the same betas (one-hot encoder also fitted on
  train metadata only).
* Chi-squared and KS tests do not feed back into the split itself.
* Test residualized data is saved alongside train residualized data,
  but ONLY using train-fit betas, never refit on test.

Outputs are written into:
    Outputs/stage2_split_stratum_aware_ESLS/
"""

import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency, ks_2samp
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

# ----------------------------- Configuration --------------------------------
PIPELINE_DIR    = Path(r"C:\Users\hafsa\Python PD Project\MI_BaggedLASSO Pipeline")
INPUT_VST       = PIPELINE_DIR / "ES vs LS" / "vst_stratum_aware_ESLS.csv"
METADATA_PATH   = PIPELINE_DIR / "ES vs LS" / "Meta_data_ESLS.csv"
OUTPUT_DIR      = PIPELINE_DIR / "ES vs LS" / "Outputs" / "stage2_split_stratum_aware_ESLS"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED            = 42
TEST_SIZE       = 0.30
CONDITION_COL   = "condition"
# cell_type is the priority confounder. Sex is included in the composite
# stratifier when it produces non-singleton strata; otherwise dropped.
PRIMARY_CONFOUNDER = "cell_type"
CONFOUNDER_COLS = ["sex", "cell_type"]
N_PCS_KS        = 5

np.random.seed(SEED)


# ------------------------------- Helpers ------------------------------------
def load_vst(path: Path) -> pd.DataFrame:
    """Load VST matrix; ensure orientation = (genes x samples)."""
    df = pd.read_csv(path, index_col=0)
    # Heuristic: ES vs LS has 102 samples; if the row count equals the
    # sample count (i.e. samples are mistakenly indexed as rows), transpose.
    if df.shape[0] == 102 and df.shape[1] != 102:
        df = df.T
    return df  # genes x samples


def align_metadata(meta: pd.DataFrame, sample_ids) -> pd.DataFrame:
    meta = meta.copy()
    if "sample_id" in meta.columns:
        meta = meta.set_index("sample_id")
    if CONDITION_COL not in meta.columns and "Class_Label" in meta.columns:
        meta = meta.rename(columns={"Class_Label": CONDITION_COL})
    needed = [CONDITION_COL] + CONFOUNDER_COLS
    missing = [c for c in needed if c not in meta.columns]
    if missing:
        raise KeyError(f"Metadata missing required columns: {missing}")
    return meta.loc[sample_ids, needed]


def compute_chi2(meta_all, meta_train, meta_test):
    variables = [CONDITION_COL] + [c for c in CONFOUNDER_COLS
                                   if c in meta_all.columns]
    rows = {}
    for var in variables:
        cats = sorted(meta_all[var].unique())
        ct = pd.DataFrame({
            "train": meta_train[var].value_counts().reindex(cats, fill_value=0),
            "test":  meta_test[var].value_counts().reindex(cats, fill_value=0),
        })
        chi2, p, dof, _ = chi2_contingency(ct)
        rows[var] = {"chi2": chi2, "p": p, "dof": int(dof)}
    return rows


def save_chi2_csv(chi2_results, out_path):
    pd.DataFrame([
        {"variable": v, "chi2": r["chi2"], "dof": r["dof"],
         "p_value": r["p"], "balanced_at_0.05": r["p"] > 0.05}
        for v, r in chi2_results.items()
    ]).to_csv(out_path, index=False)


# --------------------------- Stratification plot ----------------------------
def plot_stratification(meta_all, meta_train, meta_test,
                        chi2_results, out_path):
    """3x3 figure: marginals, composites, summary table, cross-tabs."""
    variables = [CONDITION_COL] + [c for c in CONFOUNDER_COLS
                                   if c in meta_all.columns]
    confounders = [c for c in CONFOUNDER_COLS if c in meta_all.columns]

    fig = plt.figure(figsize=(20, 17))
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.38)

    palette = {"Overall": "#95a5a6", "Train": "#2ecc71", "Test": "#e74c3c"}
    splits  = {"Overall": meta_all, "Train": meta_train, "Test": meta_test}

    # Row 1: marginal proportions, with chi^2 in the title.
    for col_idx, var in enumerate(variables[:3]):
        ax = fig.add_subplot(gs[0, col_idx])
        cats = sorted(meta_all[var].unique())
        x = np.arange(len(cats))
        width = 0.25
        offsets = [-width, 0, width]
        for s_idx, (split_name, m_s) in enumerate(splits.items()):
            counts = m_s[var].value_counts().reindex(cats, fill_value=0)
            props = counts / max(1, len(m_s)) * 100
            bars = ax.bar(x + offsets[s_idx], props.values, width,
                          label=f"{split_name} (n={len(m_s)})",
                          color=palette[split_name],
                          edgecolor="white", linewidth=0.5, alpha=0.88)
            for bar, cnt in zip(bars, counts.values):
                if bar.get_height() > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            bar.get_height() / 2, f"n={cnt}",
                            ha="center", va="center",
                            fontsize=6.5, color="white", fontweight="bold")
        r = chi2_results[var]
        sig = (f"OK  p={r['p']:.3f}" if r["p"] > 0.05
               else f"WARN p={r['p']:.3f}")
        ax.set_title(f"{var.replace('_', ' ').title()}\n"
                     f"Chi^2 train vs test: {sig}",
                     fontsize=10, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(cats, fontsize=9)
        ax.set_ylabel("Proportion (%)", fontsize=9)
        ax.set_ylim(0, max(ax.get_ylim()[1] * 1.15, 15))
        ax.legend(fontsize=7.5, loc="upper right")
        ax.grid(axis="y", alpha=0.25, linestyle=":")
        sns.despine(ax=ax)

    # Row 2 panels 0,1: composite condition × confounder stacked bars.
    for conf_idx, conf_var in enumerate(confounders[:2]):
        ax = fig.add_subplot(gs[1, conf_idx])
        cond_cats = sorted(meta_all[CONDITION_COL].unique())
        conf_cats = sorted(meta_all[conf_var].unique())
        cmap = dict(zip(conf_cats, sns.color_palette("Set2", len(conf_cats))))

        x_ticks, x_labels, current_x = [], [], 0
        for split_name, m_s in [("Train", meta_train), ("Test", meta_test)]:
            for cond in cond_cats:
                sub = m_s[m_s[CONDITION_COL] == cond]
                bottom = 0
                for cv in conf_cats:
                    cnt = (sub[conf_var] == cv).sum()
                    prop = cnt / len(sub) * 100 if len(sub) > 0 else 0
                    ax.bar(current_x, prop, 1, bottom=bottom,
                           color=cmap[cv], edgecolor="white",
                           linewidth=0.4, alpha=0.88)
                    if prop > 8:
                        ax.text(current_x, bottom + prop / 2,
                                f"{prop:.0f}%", ha="center", va="center",
                                fontsize=6, color="white", fontweight="bold")
                    bottom += prop
                x_ticks.append(current_x)
                x_labels.append(f"{cond}\n({split_name})")
                current_x += 1
            current_x += 0.6

        handles = [plt.Rectangle((0, 0), 1, 1, color=cmap[c], label=c)
                   for c in conf_cats]
        ax.legend(handles=handles, title=conf_var.replace("_", " ").title(),
                  fontsize=7.5, title_fontsize=8, loc="lower right")
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(x_labels, fontsize=7.5)
        ax.set_ylabel("Proportion within condition (%)", fontsize=9)
        ax.set_ylim(0, 115)
        ax.set_title(f"Condition × {conf_var.replace('_', ' ').title()}\n"
                     f"(stacked proportions)",
                     fontsize=10, fontweight="bold")
        ax.grid(axis="y", alpha=0.25, linestyle=":")
        sns.despine(ax=ax)

    # Row 2 panel 2: chi^2 summary table.
    ax_tbl = fig.add_subplot(gs[1, 2])
    ax_tbl.axis("off")
    rows = [["Variable", "Chi^2", "df", "p-value", "Balanced?"]]
    for var in variables:
        r = chi2_results[var]
        rows.append([var.replace("_", " ").title(),
                     f"{r['chi2']:.3f}", str(r["dof"]),
                     f"{r['p']:.4f}",
                     "Yes" if r["p"] > 0.05 else "WARN"])
    tbl = ax_tbl.table(cellText=rows[1:], colLabels=rows[0],
                       cellLoc="center", loc="center",
                       bbox=[0, 0.2, 1, 0.75])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    for (rr, cc), cell in tbl.get_celld().items():
        cell.set_edgecolor("#cccccc")
        if rr == 0:
            cell.set_facecolor("#2c3e50")
            cell.set_text_props(color="white", fontweight="bold")
        elif "Yes" in cell.get_text().get_text():
            cell.set_facecolor("#d5f5e3")
        elif "WARN" in cell.get_text().get_text():
            cell.set_facecolor("#fadbd8")
        else:
            cell.set_facecolor("#f8f9fa" if rr % 2 == 0 else "white")
    ax_tbl.set_title("Chi-squared Test: Train vs Test\n"
                     "(H0: distributions are equal)",
                     fontsize=10, fontweight="bold", pad=12)

    # Row 3: cross-tab heatmaps.
    pairs = [(CONDITION_COL, v) for v in confounders[:2]]
    splits_3 = [("Train", meta_train), ("Test", meta_test)]
    positions = [(2, 0), (2, 1), (2, 2)]
    h_idx = 0
    for row_var, col_var in pairs:
        for split_name, m_s in splits_3:
            if h_idx >= len(positions):
                break
            ax = fig.add_subplot(gs[positions[h_idx]])
            ct = pd.crosstab(m_s[row_var], m_s[col_var])
            sns.heatmap(ct, annot=True, fmt="d", cmap="Blues",
                        ax=ax, linewidths=0.5, linecolor="white",
                        cbar_kws={"shrink": 0.75})
            ax.set_title(f"{row_var.replace('_', ' ').title()} × "
                         f"{col_var.replace('_', ' ').title()}\n"
                         f"({split_name})",
                         fontsize=9.5, fontweight="bold")
            ax.set_xlabel(col_var.replace("_", " ").title(), fontsize=8.5)
            ax.set_ylabel(row_var.replace("_", " ").title(), fontsize=8.5)
            ax.tick_params(labelsize=8)
            h_idx += 1

    fig.suptitle(
        "Stage 2 (ES vs LS) — Composite Stratification: Train / Test "
        "Distribution\n"
        f"(70/30 split, stratified by condition + cell_type [+ sex if "
        f"feasible])",
        fontsize=13, fontweight="bold", y=1.01)

    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()


# --------------------------- Variance partition -----------------------------
def variance_partition(Y, meta_block):
    """
    Per-gene partial R^2 (Type II) for each predictor in meta_block.
    Returns dict {col -> np.ndarray(n_genes), 'residual' -> np.ndarray}.
    """
    cols = [CONDITION_COL] + [c for c in CONFOUNDER_COLS
                              if c in meta_block.columns]
    enc = OneHotEncoder(drop="first", sparse_output=False,
                        handle_unknown="ignore")
    blocks, parts, cursor = {}, [np.ones((len(meta_block), 1))], 1
    for col in cols:
        d = enc.fit_transform(meta_block[[col]])
        blocks[col] = (cursor, cursor + d.shape[1])
        parts.append(d)
        cursor += d.shape[1]
    X = np.hstack(parts)

    Y_mean = Y.mean(axis=0, keepdims=True)
    SS_tot = ((Y - Y_mean) ** 2).sum(axis=0) + 1e-10

    def ss_res(Xd):
        B = np.linalg.lstsq(Xd, Y, rcond=None)[0]
        return ((Y - Xd @ B) ** 2).sum(axis=0)

    SS_full = ss_res(X)
    out = {}
    for col in cols:
        s, e = blocks[col]
        Xr = np.hstack([X[:, :s], X[:, e:]])
        out[col] = np.clip((ss_res(Xr) - SS_full) / SS_tot, 0, 1)
    out["residual"] = np.clip(SS_full / SS_tot, 0, 1)
    return out


def fit_mirror_ols(X_train, meta_train, conf_cols):
    """
    Fit OLS X ~ confounders on TRAIN. Return a closure that residualizes
    any (X, meta) pair using the same encoder + betas. Pure mirror — no
    refit on test data.
    """
    enc = OneHotEncoder(drop="first", sparse_output=False,
                        handle_unknown="ignore")
    enc.fit(meta_train[conf_cols])

    def design_matrix(meta_block):
        D = enc.transform(meta_block[conf_cols])
        return np.column_stack([np.ones(len(meta_block)), D])

    C_train = design_matrix(meta_train)
    B_ols = np.linalg.pinv(C_train) @ X_train

    def residualize(X, meta_block):
        C = design_matrix(meta_block)
        return X - C[:, 1:] @ B_ols[1:, :]

    return residualize, B_ols, enc


def plot_variance_partitioning(r2_before, r2_after, n_genes, out_path):
    predictors = [CONDITION_COL] + [c for c in CONFOUNDER_COLS
                                    if c in r2_before]
    cols = predictors + ["residual"]
    n_pred = len(cols)

    color_map = {
        CONDITION_COL: "#2980b9",
        "sex":         "#e74c3c",
        "cell_type":   "#e67e22",
        "residual":    "#7f8c8d",
    }

    fig, axes = plt.subplots(2, n_pred, figsize=(3.5 * n_pred, 7),
                             sharey=True, constrained_layout=True)
    row_labels = ["Before residualization", "After residualization"]
    for row, (r2_dict, lbl) in enumerate(zip([r2_before, r2_after],
                                             row_labels)):
        for col, var in enumerate(cols):
            ax = axes[row, col]
            data = r2_dict.get(var, np.zeros(n_genes)) * 100
            color = color_map.get(var, "#95a5a6")
            parts = ax.violinplot(data, positions=[0],
                                  showmedians=True, showextrema=True,
                                  widths=0.7)
            for pc in parts["bodies"]:
                pc.set_facecolor(color)
                pc.set_alpha(0.55)
            parts["cmedians"].set_color("black")
            parts["cmedians"].set_linewidth(2)

            med = float(np.median(data))
            ax.text(0.08, med, f" {med:.1f}%", va="center",
                    fontsize=8.5, color="black", fontweight="bold",
                    transform=ax.get_yaxis_transform())
            title = (var.replace("_", " ").title()
                     if var != "residual" else "Unexplained")
            if row == 0:
                ax.set_title(title, fontsize=10, fontweight="bold",
                             color=color)
            ax.set_xticks([])
            if col == 0:
                ax.set_ylabel(f"{lbl}\nPartial R^2 (%)", fontsize=8.5)
            ax.set_ylim(-2, 105)
            ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
            sns.despine(ax=ax)

    fig.suptitle(
        "Variance Partitioning (ES vs LS): Per-Gene Partial R^2 — "
        "Before vs After Mirror OLS Residualization\n"
        "(cell_type / sex R^2 should drop near 0 after residualization; "
        "Condition R^2 should stay)",
        fontsize=11, fontweight="bold")
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()


# ----------------------------- Main pipeline --------------------------------
def main():
    print("=" * 70)
    print("STAGE 2 (ES vs LS) — Stratified 70/30 split "
          "(vst_stratum_aware_ESLS.csv)")
    print("=" * 70)

    # Load
    vst = load_vst(INPUT_VST)
    sample_ids = vst.columns.tolist()
    gene_names = vst.index.tolist()
    X_full = vst.T.values  # (n_samples, n_genes)

    meta = align_metadata(pd.read_csv(METADATA_PATH), sample_ids)
    y = meta[CONDITION_COL].values

    print(f"\nLoaded VST: {X_full.shape[0]} samples x "
          f"{X_full.shape[1]} genes")
    print(f"Class balance: "
          f"{pd.Series(y).value_counts().to_dict()}")
    print(f"Sex balance:   "
          f"{meta['sex'].value_counts().to_dict()}")
    print(f"Cell types:    "
          f"{meta['cell_type'].value_counts().to_dict()}")

    # Composite stratification key with cell_type as priority confounder.
    # Try (condition x cell_type x sex) first; fall back to
    # (condition x cell_type) on singletons (likely given F=20).
    strat_key = (meta[CONDITION_COL].astype(str)
                 + "__" + meta[PRIMARY_CONFOUNDER].astype(str)
                 + "__" + meta["sex"].astype(str))
    counts = strat_key.value_counts()
    print(f"\nComposite strata cond x cell_type x sex (n={len(counts)}):")
    print(counts.to_string())

    if (counts == 1).any():
        print(f"  [WARN] {(counts == 1).sum()} composite strata have n=1.")
        print(f"         Falling back to (condition x cell_type) — "
              f"cell_type is the priority confounder.")
        strat_key = (meta[CONDITION_COL].astype(str)
                     + "__" + meta[PRIMARY_CONFOUNDER].astype(str))
        counts = strat_key.value_counts()
        print(counts.to_string())
        if (counts == 1).any():
            print(f"  [WARN] still has singletons; falling back to "
                  f"condition only.")
            strat_key = meta[CONDITION_COL].astype(str)

    # Split
    sss = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE,
                                 random_state=SEED)
    train_idx, test_idx = next(sss.split(X_full, strat_key))

    X_train = X_full[train_idx]
    X_test  = X_full[test_idx]
    y_train = y[train_idx]
    y_test  = y[test_idx]
    meta_train = meta.iloc[train_idx].copy()
    meta_test  = meta.iloc[test_idx].copy()
    train_ids = [sample_ids[i] for i in train_idx]
    test_ids  = [sample_ids[i] for i in test_idx]

    print(f"\nTrain (n={len(y_train)}):")
    print(pd.Series(y_train).value_counts().to_string())
    print(f"\nTest  (n={len(y_test)}):")
    print(pd.Series(y_test).value_counts().to_string())

    # ---- (a) Chi-squared train vs test ----
    chi2_results = compute_chi2(meta, meta_train, meta_test)
    save_chi2_csv(chi2_results, OUTPUT_DIR / "chi_squared.csv")
    print("\nChi-squared (train vs test):")
    for v, r in chi2_results.items():
        flag = "OK" if r["p"] > 0.05 else "WARN"
        print(f"  {v:12s} chi2={r['chi2']:.3f} df={r['dof']} "
              f"p={r['p']:.4f}  [{flag}]")
    plot_stratification(meta, meta_train, meta_test, chi2_results,
                        OUTPUT_DIR / "stratification.png")

    # ---- (b) KS-test on PCs (PCA fit on TRAIN ONLY, no leakage) ----
    n_pcs = min(N_PCS_KS, min(X_train.shape) - 1)
    pca = PCA(n_components=n_pcs, random_state=SEED)
    pca.fit(X_train)
    pcs_train = pca.transform(X_train)
    pcs_test  = pca.transform(X_test)

    ks_rows = []
    for i in range(n_pcs):
        ks, p = ks_2samp(pcs_train[:, i], pcs_test[:, i])
        ks_rows.append({
            "PC": f"PC{i + 1}",
            "var_explained": pca.explained_variance_ratio_[i],
            "KS_stat": ks, "p_value": p,
            "balanced_at_0.05": p > 0.05,
        })
    pd.DataFrame(ks_rows).to_csv(OUTPUT_DIR / "ks_test_on_pcs.csv",
                                 index=False)
    print(f"\nKS-test on top-{n_pcs} TRAIN-fit PCs (train vs test):")
    for r in ks_rows:
        flag = "OK" if r["balanced_at_0.05"] else "WARN"
        print(f"  {r['PC']}: var_exp={r['var_explained']:.1%}, "
              f"KS={r['KS_stat']:.4f}, p={r['p_value']:.4f}  [{flag}]")

    # ---- (c) Variance partitioning before / after mirror OLS ----
    conf_cols = [c for c in CONFOUNDER_COLS if c in meta_train.columns]
    print("\nFitting mirror OLS on TRAIN (confounders = "
          f"{conf_cols}) ...")
    residualize, B_ols, _enc = fit_mirror_ols(X_train, meta_train, conf_cols)

    # Apply mirror residualization (TRAIN with TRAIN-fit, TEST with TRAIN-fit)
    X_train_resid = residualize(X_train, meta_train)
    X_test_resid  = residualize(X_test,  meta_test)

    print("Computing per-gene partial R^2 (this can take a moment)...")
    r2_before = variance_partition(X_train,       meta_train)
    r2_after  = variance_partition(X_train_resid, meta_train)

    # Save partial R^2 per gene as a wide table
    r2_table = pd.DataFrame({"gene": gene_names})
    for k, v in r2_before.items():
        r2_table[f"{k}_before"] = v
    for k, v in r2_after.items():
        r2_table[f"{k}_after"] = v
    r2_table.to_csv(OUTPUT_DIR / "variance_partition_per_gene.csv",
                    index=False)

    plot_variance_partitioning(r2_before, r2_after, X_train.shape[1],
                               OUTPUT_DIR / "variance_partitioning.png")

    print("Median partial R^2 (% across genes):")
    for var in [CONDITION_COL] + conf_cols + ["residual"]:
        before = float(np.median(r2_before[var])) * 100
        after  = float(np.median(r2_after[var]))  * 100
        print(f"  {var:12s} before={before:5.1f}%   after={after:5.1f}%")

    # ---- Save split + residualized data ----
    print("\nSaving outputs ...")
    meta_train.to_csv(OUTPUT_DIR / "meta_train.csv")
    meta_test.to_csv(OUTPUT_DIR / "meta_test.csv")

    train_df = pd.DataFrame(X_train.T, index=gene_names, columns=train_ids)
    train_df.index.name = "gene"
    train_df.to_csv(OUTPUT_DIR / "train.csv")

    test_df = pd.DataFrame(X_test.T, index=gene_names, columns=test_ids)
    test_df.index.name = "gene"
    test_df.to_csv(OUTPUT_DIR / "test.csv")

    train_resid_df = pd.DataFrame(X_train_resid.T,
                                  index=gene_names, columns=train_ids)
    train_resid_df.index.name = "gene"
    train_resid_df.to_csv(OUTPUT_DIR / "train_residualized.csv")

    test_resid_df = pd.DataFrame(X_test_resid.T,
                                 index=gene_names, columns=test_ids)
    test_resid_df.index.name = "gene"
    test_resid_df.to_csv(OUTPUT_DIR / "test_residualized.csv")

    pd.DataFrame(B_ols, columns=gene_names).to_csv(
        OUTPUT_DIR / "mirror_ols_betas.csv", index_label="design_row")

    rows = []
    for split_name, m_s in [("Overall", meta), ("Train", meta_train),
                            ("Test", meta_test)]:
        for var in [CONDITION_COL] + CONFOUNDER_COLS:
            counts = m_s[var].value_counts().sort_index()
            for cat, cnt in counts.items():
                rows.append({"split": split_name, "variable": var,
                             "category": cat,
                             "count": int(cnt),
                             "total": len(m_s),
                             "proportion": round(cnt / len(m_s) * 100, 2)})
    pd.DataFrame(rows).to_csv(OUTPUT_DIR / "split_summary.csv", index=False)

    print(f"\nAll outputs in: {OUTPUT_DIR}")
    print(" - stratification.png        chi-squared visualisation")
    print(" - chi_squared.csv           chi-squared statistics per variable")
    print(" - ks_test_on_pcs.csv        KS test (train vs test) on top PCs")
    print(" - variance_partitioning.png partial R^2 violin plot")
    print(" - variance_partition_per_gene.csv  partial R^2 table per gene")
    print(" - mirror_ols_betas.csv      OLS coefficients (train-fit)")
    print(" - train.csv / test.csv      raw VST splits")
    print(" - train_residualized.csv / test_residualized.csv  mirror-OLS residuals")
    print(" - meta_train.csv / meta_test.csv  metadata for each split")
    print(" - split_summary.csv         counts and proportions")


if __name__ == "__main__":
    main()


STAGE 2 (ES vs LS) — Stratified 70/30 split (vst_stratum_aware_ESLS.csv)

Loaded VST: 102 samples x 7446 genes
Class balance: {'ES': 60, 'LS': 42}
Sex balance:   {'M': 82, 'F': 20}
Cell types:    {'CD8': 36, 'CD4': 35, 'PBMC': 31}

Composite strata cond x cell_type x sex (n=12):
ES__CD8__M     18
ES__CD4__M     17
ES__PBMC__M    15
LS__PBMC__M    11
LS__CD8__M     11
LS__CD4__M     10
LS__CD4__F      4
ES__CD4__F      4
LS__CD8__F      4
ES__PBMC__F     3
ES__CD8__F      3
LS__PBMC__F     2

Train (n=71):
ES    41
LS    30

Test  (n=31):
ES    19
LS    12

Chi-squared (train vs test):
  condition    chi2=0.013 df=1 p=0.9078  [OK]
  sex          chi2=0.000 df=1 p=1.0000  [OK]
  cell_type    chi2=0.106 df=2 p=0.9482  [OK]

KS-test on top-5 TRAIN-fit PCs (train vs test):
  PC1: var_exp=15.5%, KS=0.1081, p=0.9319  [OK]
  PC2: var_exp=10.7%, KS=0.1145, p=0.8998  [OK]
  PC3: var_exp=7.6%, KS=0.1286, p=0.8110  [OK]
  PC4: var_exp=6.8%, KS=0.1427, p=0.7052  [OK]
  PC5: var_exp=4.7%, KS=0.1686,